# NETI-HyOptima Phase 1: Core Model Demonstration

This notebook demonstrates the HyOptima optimization engine for hybrid energy systems.

## Overview

NETI-HyOptima (Nigeria Energy Transition Intelligence Platform) is a decision intelligence platform designed to convert Nigeria's Energy Transition Plan into actionable, bankable investment decisions.

The **HyOptima Core Engine** solves the hybrid energy optimization problem:
- Minimize total system cost (investment + operating)
- Meet electricity demand reliably
- Optimize solar, gas, and battery storage mix
- Align with Nigeria's energy transition goals

## Model Components

1. **Load Profile**: Electricity demand over time
2. **Solar Profile**: Solar availability/capacity factor
3. **Economic Parameters**: Costs (CAPEX, OPEX, fuel)
4. **Technical Parameters**: Efficiencies, limits
5. **Optimization Model**: MILP formulation

---

## 1. Setup and Imports

In [ ]:
# Standard imports
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

# HyOptima imports
from hyoptima import (
    HyOptimaModel,
    HyOptimaSolver,
    EconomicParameters,
    TechnicalParameters,
    PolicyParameters,
    LoadProfile,
    SolarProfile,
)
from hyoptima.profiles import generate_nigeria_scenarios
from hyoptima.utils import plot_dispatch_results, generate_report

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("NETI-HyOptima Phase 1: Core Model")
print("="*50)

## 2. Generate Synthetic Data

We'll create synthetic load and solar profiles representing a Nigerian community microgrid.

In [ ]:
# Generate synthetic profiles for a Nigerian community
# Example: Bauchi state rural community

load_profile = LoadProfile.generate_synthetic(
    peak_demand=300,      # 300 kW peak demand
    profile_type="mixed", # Mixed residential/commercial
    hours=24,
    noise_level=0.05,
    name="bauchi_community"
)

solar_profile = SolarProfile.from_capacity_factor(
    capacity_factor=0.20,  # 20% capacity factor (typical for Northern Nigeria)
    hours=24
)
solar_profile.name = "bauchi_solar"

# Print profile summaries
print("Load Profile Summary:")
print(f"  Peak Demand: {load_profile.peak_demand:.1f} kW")
print(f"  Total Energy: {load_profile.total_energy:.1f} kWh")
print(f"  Load Factor: {load_profile.load_factor:.1%}")
print()
print("Solar Profile Summary:")
print(f"  Capacity Factor: {solar_profile.capacity_factor:.1%}")
print(f"  Peak Sun Hours: {solar_profile.peak_hours:.1f} hours")
print(f"  Daylight Hours: {solar_profile.daylight_hours}")

In [ ]:
# Visualize the input profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Load profile
hours = range(24)
axes[0].bar(hours, load_profile.demand, color='#2196F3', alpha=0.7)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Demand (kW)')
axes[0].set_title('Load Profile - Nigerian Community')
axes[0].set_xlim(-0.5, 23.5)
axes[0].axhline(y=load_profile.peak_demand, color='r', linestyle='--', 
                label=f'Peak: {load_profile.peak_demand:.0f} kW')
axes[0].legend()

# Solar profile
axes[1].fill_between(hours, solar_profile.availability, color='#FFD700', alpha=0.7)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Availability Factor')
axes[1].set_title('Solar Availability Profile')
axes[1].set_xlim(0, 23)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/figures/input_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Configure Parameters

Set economic and technical parameters based on Nigerian market conditions.

In [ ]:
# Economic parameters (Nigeria-specific estimates)
economic_params = EconomicParameters(
    solar_capex=800,           # $/kW installed
    gas_capex=500,             # $/kW installed
    battery_capex=300,         # $/kWh capacity
    gas_fuel_cost=0.08,        # $/kWh fuel cost
    grid_tariff=0.12,          # $/kWh (if grid available)
    unserved_energy_penalty=1.0,  # High penalty for reliability
    discount_rate=0.10,        # 10% discount rate
)

# Technical parameters
technical_params = TechnicalParameters(
    battery_charge_efficiency=0.95,
    battery_discharge_efficiency=0.95,
    battery_c_rate=0.5,        # C/2 max charge/discharge
    target_reliability=0.99,   # 99% reliability target
    max_solar_capacity=2000,   # 2 MW max
    max_gas_capacity=1000,     # 1 MW max
    max_battery_capacity=4000, # 4 MWh max
)

print("Economic Parameters:")
print(f"  Solar CAPEX: ${economic_params.solar_capex}/kW")
print(f"  Gas CAPEX: ${economic_params.gas_capex}/kW")
print(f"  Battery CAPEX: ${economic_params.battery_capex}/kWh")
print(f"  Gas Fuel Cost: ${economic_params.gas_fuel_cost}/kWh")
print()
print("Technical Parameters:")
print(f"  Battery Efficiency: {technical_params.battery_charge_efficiency:.0%}")
print(f"  Target Reliability: {technical_params.target_reliability:.0%}")

## 4. Build Optimization Model

In [ ]:
# Create the optimization model
model = HyOptimaModel(
    load_profile=load_profile,
    solar_profile=solar_profile,
    economic_params=economic_params,
    technical_params=technical_params,
    name="Bauchi_Community_v0"
)

# Build the Pyomo model
model.build_model()

# Print model summary
print(model.get_model_summary())

## 5. Run Optimization

In [ ]:
# Create solver and run optimization
solver = HyOptimaSolver(
    solver_name="highs",  # HiGHS solver (fast, open-source)
    tee=True              # Print solver output
)

# Solve the model
results = solver.solve(model)

# Print results summary
print(solver.print_summary(results))

## 6. Analyze Results

In [ ]:
# Key results
print("="*60)
print("OPTIMAL CAPACITY DECISIONS")
print("="*60)
print(f"Solar Capacity:   {results['solar_capacity_kw']:.1f} kW")
print(f"Gas Capacity:     {results['gas_capacity_kw']:.1f} kW")
print(f"Battery Capacity: {results['battery_capacity_kwh']:.1f} kWh")
print()
print(f"Total System Cost: ${results['total_cost']:,.2f}")
print()
print("PERFORMANCE METRICS")
print("-"*40)
metrics = results['metrics']
print(f"Renewable Fraction: {metrics['renewable_fraction']:.1%}")
print(f"Reliability:        {metrics['reliability']:.2%}")
print(f"Approximate LCOE:   ${metrics['lcoe_approx']:.3f}/kWh")
print(f"CO2 Emissions:      {metrics['emissions_kg']:.0f} kg")

In [ ]:
# Get explanation of results
explanation = solver.get_explanation(results)
print("\n" + "="*60)
print("DECISION EXPLANATION")
print("="*60)
print(explanation)

## 7. Visualize Dispatch Results

In [ ]:
# Plot comprehensive dispatch results
fig = plot_dispatch_results(
    results,
    save_path="../results/figures/phase1_dispatch.png",
    show=True
)

In [ ]:
# Detailed dispatch analysis
dispatch = results['dispatch']
hours = range(24)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Generation stack
ax1 = axes[0, 0]
ax1.stackplot(hours, 
              dispatch['solar_gen'], 
              dispatch['gas_gen'],
              dispatch['grid_import'],
              labels=['Solar', 'Gas', 'Grid'],
              colors=['#FFD700', '#8B4513', '#808080'],
              alpha=0.7)
ax1.plot(hours, dispatch['demand'], 'k--', linewidth=2, label='Demand')
ax1.set_xlabel('Hour')
ax1.set_ylabel('Power (kW)')
ax1.set_title('Generation Stack vs Demand')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Battery operation
ax2 = axes[0, 1]
ax2.fill_between(hours, dispatch['soc'], alpha=0.5, color='green', label='SOC')
ax2_twin = ax2.twinx()
ax2_twin.plot(hours, dispatch['battery_charge'], 'b-', label='Charge', linewidth=2)
ax2_twin.plot(hours, dispatch['battery_discharge'], 'r-', label='Discharge', linewidth=2)
ax2.set_xlabel('Hour')
ax2.set_ylabel('SOC (kWh)', color='green')
ax2_twin.set_ylabel('Power (kW)')
ax2.set_title('Battery State of Charge and Power')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Plot 3: Solar utilization
ax3 = axes[1, 0]
solar_cap = results['solar_capacity_kw']
solar_available = [solar_cap * dispatch['solar_availability'][h] for h in hours]
ax3.bar(hours, solar_available, color='#FFEB3B', alpha=0.5, label='Available')
ax3.bar(hours, dispatch['solar_gen'], color='#FFD700', alpha=0.9, label='Generated')
ax3.set_xlabel('Hour')
ax3.set_ylabel('Power (kW)')
ax3.set_title('Solar Generation vs Availability')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Gas generator operation
ax4 = axes[1, 1]
ax4.bar(hours, dispatch['gas_gen'], color='#8B4513', alpha=0.7)
ax4.axhline(y=results['gas_capacity_kw'], color='r', linestyle='--', 
            label=f"Capacity: {results['gas_capacity_kw']:.0f} kW")
ax4.set_xlabel('Hour')
ax4.set_ylabel('Power (kW)')
ax4.set_title('Gas Generator Output')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/phase1_detailed_dispatch.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Scenario Comparison

Compare different Nigerian locations and scenarios.

In [ ]:
# Compare different Nigerian locations
locations = ['kano', 'lagos', 'abuja', 'bauchi']
results_by_location = {}

for location in locations:
    # Generate location-specific profiles
    load, solar = generate_nigeria_scenarios(location, season='dry')
    
    # Create and solve model
    model = HyOptimaModel(
        load_profile=load,
        solar_profile=solar,
        economic_params=economic_params,
        technical_params=technical_params,
        name=f"{location}_scenario"
    )
    
    solver = HyOptimaSolver(solver_name='highs', tee=False)
    results = solver.solve(model)
    results_by_location[location] = results
    
    print(f"\n{location.upper()}:")
    print(f"  Solar: {results['solar_capacity_kw']:.0f} kW")
    print(f"  Gas: {results['gas_capacity_kw']:.0f} kW")
    print(f"  Battery: {results['battery_capacity_kwh']:.0f} kWh")
    print(f"  Cost: ${results['total_cost']:,.0f}")
    print(f"  Renewable: {results['metrics']['renewable_fraction']:.1%}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(len(locations))
width = 0.6

# Capacity comparison
solar_caps = [results_by_location[loc]['solar_capacity_kw'] for loc in locations]
gas_caps = [results_by_location[loc]['gas_capacity_kw'] for loc in locations]
battery_caps = [results_by_location[loc]['battery_capacity_kwh'] for loc in locations]

axes[0].bar(x - 0.2, solar_caps, width=0.3, label='Solar (kW)', color='#FFD700')
axes[0].bar(x + 0.1, gas_caps, width=0.3, label='Gas (kW)', color='#8B4513')
axes[0].set_xticks(x)
axes[0].set_xticklabels([loc.title() for loc in locations])
axes[0].set_ylabel('Capacity (kW)')
axes[0].set_title('Generation Capacity by Location')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Battery capacity
axes[1].bar(x, battery_caps, width=width, color='#228B22')
axes[1].set_xticks(x)
axes[1].set_xticklabels([loc.title() for loc in locations])
axes[1].set_ylabel('Capacity (kWh)')
axes[1].set_title('Battery Capacity by Location')
axes[1].grid(True, alpha=0.3, axis='y')

# Cost and renewable fraction
costs = [results_by_location[loc]['total_cost'] for loc in locations]
renewables = [results_by_location[loc]['metrics']['renewable_fraction'] for loc in locations]

ax3_twin = axes[2].twinx()
bars1 = axes[2].bar(x - 0.2, costs, width=0.35, color='#2196F3', label='Total Cost ($)')
bars2 = ax3_twin.bar(x + 0.2, renewables, width=0.35, color='#4CAF50', label='Renewable Fraction')

axes[2].set_xticks(x)
axes[2].set_xticklabels([loc.title() for loc in locations])
axes[2].set_ylabel('Cost ($)', color='#2196F3')
ax3_twin.set_ylabel('Renewable Fraction', color='#4CAF50')
axes[2].set_title('Cost and Renewable Fraction')
axes[2].legend(loc='upper left')
ax3_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../results/figures/location_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sensitivity Analysis

Analyze how results change with different parameter values.

In [ ]:
# Sensitivity to fuel cost
fuel_costs = [0.04, 0.06, 0.08, 0.10, 0.12, 0.15]
sensitivity_results = []

for fuel_cost in fuel_costs:
    econ = EconomicParameters(gas_fuel_cost=fuel_cost)
    model = HyOptimaModel(load_profile, solar_profile, economic_params=econ)
    solver = HyOptimaSolver(solver_name='highs', tee=False)
    result = solver.solve(model)
    sensitivity_results.append(result)

# Plot sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Capacity vs fuel cost
axes[0].plot(fuel_costs, [r['solar_capacity_kw'] for r in sensitivity_results], 
             'o-', color='#FFD700', linewidth=2, markersize=8, label='Solar')
axes[0].plot(fuel_costs, [r['gas_capacity_kw'] for r in sensitivity_results], 
             'o-', color='#8B4513', linewidth=2, markersize=8, label='Gas')
axes[0].set_xlabel('Gas Fuel Cost ($/kWh)')
axes[0].set_ylabel('Capacity (kW)')
axes[0].set_title('Capacity vs Fuel Cost')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cost and renewable fraction vs fuel cost
ax_twin = axes[1].twinx()
axes[1].plot(fuel_costs, [r['total_cost'] for r in sensitivity_results], 
             'o-', color='#2196F3', linewidth=2, markersize=8, label='Total Cost')
ax_twin.plot(fuel_costs, [r['metrics']['renewable_fraction'] for r in sensitivity_results], 
             's-', color='#4CAF50', linewidth=2, markersize=8, label='Renewable Fraction')

axes[1].set_xlabel('Gas Fuel Cost ($/kWh)')
axes[1].set_ylabel('Total Cost ($)', color='#2196F3')
ax_twin.set_ylabel('Renewable Fraction', color='#4CAF50')
axes[1].set_title('Cost and Renewable Fraction vs Fuel Cost')
axes[1].legend(loc='upper left')
ax_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../results/figures/sensitivity_fuel_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Results

In [ ]:
# Save main results
solver.save_results(results, '../results/runs/phase1_results.json')

# Generate and save report
report = generate_report(results, template='detailed')
with open('../results/runs/phase1_report.txt', 'w') as f:
    f.write(report)

print("Results saved to:")
print("  - ../results/runs/phase1_results.json")
print("  - ../results/runs/phase1_report.txt")
print("  - ../results/figures/phase1_dispatch.png")
print("  - ../results/figures/phase1_detailed_dispatch.png")

## 11. Summary and Conclusions

In [ ]:
print("\n" + "="*70)
print("NETI-HyOptima Phase 1: Summary")
print("="*70)

print("""
The HyOptima Core Engine has been successfully demonstrated for a 
Nigerian community microgrid scenario. Key findings:

1. OPTIMAL SYSTEM CONFIGURATION
   - The model determines the cost-optimal mix of solar, gas, and battery
   - Battery storage enables higher solar penetration
   - Gas provides reliability during low-solar periods

2. COST-EFFICIENCY
   - Total system cost is minimized while meeting reliability targets
   - LCOE provides a benchmark for comparison with grid alternatives

3. RENEWABLE INTEGRATION
   - Significant renewable fraction achieved (varies by location)
   - Northern Nigeria (higher solar resource) shows better economics

4. POLICY ALIGNMENT
   - Results support Nigeria's Energy Transition Plan goals
   - Gas serves as transition fuel while renewable capacity scales

NEXT STEPS (Phase 2):
- Add stochastic optimization for uncertainty handling
- Extend to multi-day and seasonal optimization
- Include emission constraints and policy targets
- Develop data pipelines for real Nigerian data
""")

print("="*70)
print("Phase 1 Complete: HyOptima v0 validated and operational")
print("="*70)